#**Trabalho 4 - K-means - Tratamento de dados**

In [ ]:
# ============================================================
# TRABALHO 4 – Kmeans (ADULT INCOME)
# Fluxo completo: dados -> recategorização/engenharia -> EDA -> modelos -> comparação
# ============================================================

# 1. Imports e setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# SMOTE
from imblearn.over_sampling import SMOTE

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 2. Leitura da base
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
caminho = "/content/drive/MyDrive/Seletivo_IPEA_2025/Trabalho3-RedesNeurais/adult.csv"
df_raw = pd.read_csv(caminho)
# Remover espaços e padronizar strings
for col in df_raw.select_dtypes(include="object").columns:
    df_raw[col] = df_raw[col].astype(str).str.strip()

# Substituir qualquer formato de interrogação por NaN
df_raw = df_raw.replace(to_replace=r"^\?$", value=np.nan, regex=True)
df_raw = df_raw.replace(to_replace=r"^\?\?$", value=np.nan, regex=True)
df_raw = df_raw.replace("?", np.nan)
df_raw = df_raw.replace(" ?", np.nan)
df_raw = df_raw.replace("? ", np.nan)
# Padronizar strings (antes de qualquer coisa)
for col in df_raw.select_dtypes(include="object").columns:
    df_raw[col] = df_raw[col].astype(str).str.strip()

# Substituir ? por NaN
df_raw.replace("?", np.nan, inplace=True)

# Manter uma cópia da base original (antes de dropna/engenharia)
df_original = df_raw.copy()

# Remover linhas com NA para baseline
df_original = df_original.dropna().reset_index(drop=True)

# ============================================================
# 3. Recategorização + Engenharia de Features (base adaptada)
# ============================================================
df_adapt = df_original.copy()

# 3.1 workclass -> pt + raras <1% em "Outros"
map_workclass = {
    "?": "Desconhecido",
    "Federal-gov": "Governo",
    "Local-gov": "Governo",
    "State-gov": "Governo",
    "Private": "Setor Privado",
    "Self-emp-inc": "Autônomo",
    "Self-emp-not-inc": "Autônomo",
    "Without-pay": "Desempregado",
    "Never-worked": "Desempregado"
}
df_adapt["workclass_pt"] = df_adapt["workclass"].map(map_workclass)

freq_wc = df_adapt["workclass_pt"].value_counts(normalize=True)
raras_wc = freq_wc[freq_wc < 0.01].index.tolist()
df_adapt["workclass_grp"] = df_adapt["workclass_pt"].replace({c: "Outros" for c in raras_wc})

# 3.2 education -> pt + 4 grupos
map_edu = {
    "Preschool": "Pré-escola",
    "1st-4th": "1ª-4ª série",
    "5th-6th": "5ª-6ª série",
    "7th-8th": "7ª-8ª série",
    "9th": "9º ano",
    "10th": "10º ano",
    "11th": "11º ano",
    "12th": "12º ano",
    "HS-grad": "Ensino Médio",
    "Some-college": "Alguma faculdade",
    "Assoc-voc": "Tecnólogo vocacional",
    "Assoc-acdm": "Tecnólogo acadêmico",
    "Bachelors": "Bacharelado",
    "Masters": "Mestrado",
    "Prof-school": "Profissionalizante",
    "Doctorate": "Doutorado"
}
df_adapt["education_pt"] = df_adapt["education"].map(map_edu)

df_adapt["education_4grp"] = df_adapt["education_pt"].replace({
    "Pré-escola": "Ensino Fundamental",
    "1ª-4ª série": "Ensino Fundamental",
    "5ª-6ª série": "Ensino Fundamental",
    "7ª-8ª série": "Ensino Fundamental",
    "9º ano": "Ensino Fundamental",
    "10º ano": "Ensino Fundamental",
    "11º ano": "Ensino Fundamental",
    "12º ano": "Ensino Médio",
    "Ensino Médio": "Ensino Médio",
    "Tecnólogo vocacional": "Ensino Médio",
    "Tecnólogo acadêmico": "Ensino Médio",
    "Profissionalizante": "Ensino Médio",
    "Alguma faculdade": "Ensino Superior",
    "Bacharelado": "Ensino Superior",
    "Mestrado": "Pós Graduação",
    "Doutorado": "Pós Graduação"
})

# 3.3 marital.status -> pt + 3 grupos
map_marital = {
    "Married-civ-spouse": "Casado(a)",
    "Married-AF-spouse": "Casado(a) (Forças Armadas)",
    "Married-spouse-absent": "Cônjuge ausente",
    "Never-married": "Nunca casado(a)",
    "Divorced": "Divorciado(a)",
    "Separated": "Separado(a)",
    "Widowed": "Viúvo(a)"
}
df_adapt["marital_pt"] = df_adapt["marital.status"].map(map_marital)

df_adapt["marital_3grp"] = df_adapt["marital_pt"].replace({
    "Casado(a)": "Casados",
    "Casado(a) (Forças Armadas)": "Casados",
    "Cônjuge ausente": "Casados",
    "Nunca casado(a)": "Solteiros",
    "Divorciado(a)": "Divorciado(a)/Separado/Viuvo",
    "Separado(a)": "Divorciado(a)/Separado/Viuvo",
    "Viúvo(a)": "Divorciado(a)/Separado/Viuvo"
})

# 3.4 occupation -> 3 grupos
df_adapt["occupation_3grp"] = df_adapt["occupation"].replace({
    "Exec-managerial": "Profissionais/Alta Qualificação",
    "Prof-specialty": "Profissionais/Alta Qualificação",
    "Tech-support": "Profissionais/Alta Qualificação",
    "Sales": "Profissionais/Alta Qualificação",

    "Adm-clerical": "Operacionais/Serviços",
    "Craft-repair": "Operacionais/Serviços",
    "Farming-fishing": "Operacionais/Serviços",
    "Handlers-cleaners": "Operacionais/Serviços",
    "Machine-op-inspct": "Operacionais/Serviços",
    "Other-service": "Operacionais/Serviços",
    "Priv-house-serv": "Operacionais/Serviços",
    "Protective-serv": "Operacionais/Serviços",
    "Transport-moving": "Operacionais/Serviços",

    "Armed-Forces": "Outros/Indefinido",
    "?": "Outros/Indefinido"
})

# 3.5 relationship -> 3 grupos
df_adapt["relationship_3grp"] = df_adapt["relationship"].replace({
    "Husband": "Chefe de família",
    "Wife": "Chefe de família",
    "Not-in-family": "Independente/Outros",
    "Unmarried": "Independente/Outros",
    "Other-relative": "Independente/Outros",
    "Own-child": "Dependente"
})

# 3.6 race -> 3 grupos
df_adapt["race_3grp"] = df_adapt["race"].replace({
    "White": "Branca",
    "Black": "Negra",
    "Amer-Indian-Eskimo": "Outras minorias",
    "Asian-Pac-Islander": "Outras minorias",
    "Other": "Outras minorias"
})

# 3.7 native.country -> continentes
continent_map = {
    "United-States": "América do Norte",
    "Canada": "América do Norte",
    "Outlying-US(Guam-USVI-etc)": "América do Norte",

    "Mexico": "América Latina",
    "?": "Desconhecido",
    "Puerto-Rico": "América Latina",
    "Cuba": "América Latina",
    "Jamaica": "América Latina",
    "Dominican-Republic": "América Latina",
    "Haiti": "América Latina",
    "Columbia": "América Latina",
    "Guatemala": "América Latina",
    "Ecuador": "América Latina",
    "Peru": "América Latina",
    "El-Salvador": "América Latina",
    "Honduras": "América Latina",
    "Nicaragua": "América Latina",
    "Trinadad&Tobago": "América Latina",
    "South": "América Latina",

    "England": "Europa",
    "Germany": "Europa",
    "Greece": "Europa",
    "Ireland": "Europa",
    "Italy": "Europa",
    "Poland": "Europa",
    "Portugal": "Europa",
    "Scotland": "Europa",
    "Yugoslavia": "Europa",
    "France": "Europa",
    "Holand-Netherlands": "Europa",
    "Hungary": "Europa",

    "China": "Ásia",
    "Japan": "Ásia",
    "India": "Ásia",
    "Iran": "Ásia",
    "Philippines": "Ásia",
    "Vietnam": "Ásia",
    "Hong": "Ásia",
    "Cambodia": "Ásia",
    "Taiwan": "Ásia",
    "Thailand": "Ásia",
    "Laos": "Ásia",
}
df_adapt["country_continent"] = df_adapt["native.country"].map(continent_map).fillna("Outros")

# 3.8 Features inspiradas no relatório RF
df_adapt["net_capital"] = df_adapt["capital.gain"] - df_adapt["capital.loss"]
df_adapt["education_hours"] = df_adapt["education.num"] * df_adapt["hours.per.week"]

# Indicadores binários simples
df_adapt["marital_status_grouped"] = df_adapt["marital_3grp"].apply(lambda x: 1 if x == "Casados" else 0)
df_adapt["workclass_grouped"] = df_adapt["workclass_grp"].replace({
    "Governo Federal": "Publico",
    "Governo Estadual": "Publico",
    "Governo Local": "Publico",
    "Setor Privado": "Privado",
    "Autônomo Incorporado": "Privado",
    "Autônomo Não-Incorporado": "Privado",
    "Sem Remuneração": "Sem-Trabalho",
    "Nunca Trabalhou": "Sem-Trabalho",
    "Desconhecido": "Desconhecido",
    "Outros": "Outros"
})

cols_ingles = [
    "workclass", "education", "marital.status",
    "occupation", "relationship", "race",
    "native.country"
]

df_adapt = df_adapt.drop(columns=cols_ingles, errors="ignore")

# ============================================================
# 3.9 Remoção de variáveis redundantes / indevidas
# ============================================================

cols_remover_redundantes = [

    # Variáveis originais já traduzidas
    "education_pt",
    "marital_pt",
    "workclass_pt",

    # Variáveis duplicadas que representam a mesma informação
    "marital_status_grouped",
    "workclass_grouped",

    # Variável estatística que não representa característica do indivíduo
    "fnlwgt"
]

# Remover colunas, ignorando se alguma não existir
df_adapt = df_adapt.drop(columns=cols_remover_redundantes, errors="ignore")

# Já removeu as variáveis originais em inglês
cols_ingles = [
    "workclass", "education", "marital.status",
    "occupation", "relationship", "race",
    "native.country"
]
df_adapt = df_adapt.drop(columns=cols_ingles, errors="ignore")



#**Trabalho 4 - K-means - Analise Descritiva**

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import data_table
    data_table.disable_dataframe_formatter()


In [ ]:
# ============================================================
# GRÁFICO SEPARADO – APENAS AGE E HOURS PER WEEK
# ============================================================

plt.figure(figsize=(14, 5))

# Idade
plt.subplot(1, 2, 1)
sns.histplot(df_original["age"], bins=30, kde=True, color="#1f77b4")
plt.title("Distribuição da Idade")
plt.xlabel("Idade (anos)")
plt.ylabel("Frequência")

# Horas semanais
plt.subplot(1, 2, 2)
sns.histplot(df_original["hours.per.week"], bins=30, kde=True, color="#1f77b4")
plt.title("Distribuição das Horas Trabalhadas por Semana")
plt.xlabel("Horas por semana")
plt.ylabel("Frequência")

plt.tight_layout()
plt.show()


In [ ]:

# Função para adicionar rótulos com pontos como separador
def add_labels(ax):
    for patch in ax.patches:
        valor = patch.get_height()
        ax.text(
            patch.get_x() + patch.get_width()/2,
            valor,
            f"{valor:,.0f}".replace(",", "."),
            ha='center', va='bottom', fontsize=9
        )

# Lista das variáveis categóricas que você quer plotar
vars_fig1 = ["sex", "income", "workclass_grp", "education_4grp"]
vars_fig2 = ["marital_3grp", "occupation_3grp", "relationship_3grp", "race_3grp"]

# Títulos editados
titulos = {
    "sex": "Distribuição por Sexo",
    "income": "Distribuição de Renda",
    "workclass_grp": "Classificação por Setor de Trabalho",
    "education_4grp": "Nível de Escolaridade Agrupado",

    "marital_3grp": "Estado Civil Agrupado",
    "occupation_3grp": "Grupo Ocupacional",
    "relationship_3grp": "Relação com o Chefe da Família",
    "race_3grp": "Distribuição por Raça",
}

# ----------------------------------------------------------------------
# FIGURA 1 (4 gráficos)
# ----------------------------------------------------------------------
plt.figure(figsize=(18, 10))
for i, col in enumerate(vars_fig1, 1):
    plt.subplot(2, 2, i)
    ax = sns.countplot(data=df_adapt, x=col, color="#1f77b4")
    add_labels(ax)
    plt.title(titulos[col], fontsize=12)
    #plt.xlabel(f"Categorias de {col}", fontsize=10)
    plt.ylabel("Frequência", fontsize=10)
    plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------
# FIGURA 2 (4 gráficos)
# ----------------------------------------------------------------------
plt.figure(figsize=(18, 10))
for i, col in enumerate(vars_fig2, 1):
    plt.subplot(2, 2, i)
    ax = sns.countplot(data=df_adapt, x=col, color="#1f77b4")
    add_labels(ax)
    plt.title(titulos[col], fontsize=12)
    #plt.xlabel(f"Categorias de {col}", fontsize=10)
    plt.ylabel("Frequência", fontsize=10)
    plt.xticks(rotation=20)

plt.tight_layout()
plt.show()


#**FASE 1 — Configuração e Pré-processamento**

In [ ]:
# ============================================================
# FASE 1 — Dataset preparado para K-means
# ============================================================

df_cluster = df_adapt.copy()

# Criar coluna binária só para INTERPRETAÇÃO (não entra no PCA/Kmeans)
df_cluster["income_bin"] = (df_cluster["income"] == ">50K").astype(int)
df_cluster = df_cluster.drop(columns=["income"], errors="ignore")

# One-hot encoding
df_encoded = pd.get_dummies(df_cluster, drop_first=True)

# X = atributos para cluster (sem renda)
X_full = df_encoded.drop(columns=["income_bin"])
y_full = df_encoded["income_bin"]

print("Shape final:", X_full.shape)


FASE 2 — SELEÇÃO INICIAL DO MELHOR k (SEM REMOVER VARIÁVEIS)
Objetivo corporativo: Determinar o “k natural” dos dados antes de qualquer limpeza (tipicamente entre 14 e 22 clusters nesta base).

In [ ]:
# ============================================================
# FASE 2 — Seleção inicial do número de clusters
# ============================================================

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Escalonar
scaler = StandardScaler()
X_scaled_full = scaler.fit_transform(X_full)

# PCA 90%
pca = PCA(n_components=0.90)
X_pca_full = pca.fit_transform(X_scaled_full)

print("Formato PCA:", X_pca_full.shape)

# Testar k de 2 até 20 (flexível)
ks = range(2, 21)

sil_vals = []
dbi_vals = []
ch_vals = []
sse_vals = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = km.fit_predict(X_pca_full)

    sil_vals.append(silhouette_score(X_pca_full, labels))
    dbi_vals.append(davies_bouldin_score(X_pca_full, labels))
    ch_vals.append(calinski_harabasz_score(X_pca_full, labels))
    sse_vals.append(km.inertia_)

df_metrics_full = pd.DataFrame({
    "k": ks,
    "Silhouette": sil_vals,
    "DBI": dbi_vals,
    "CH": ch_vals,
    "SSE": sse_vals
})

df_metrics_full


2.B — Visualização dos gráficos

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(14,10))

# Silhouette
axs[0,0].plot(ks, sil_vals, marker='o')
axs[0,0].set_title("Silhouette")
axs[0,0].grid(True)

# DBI
axs[0,1].plot(ks, dbi_vals, marker='o')
axs[0,1].set_title("DBI (menor é melhor)")
axs[0,1].grid(True)

# CH
axs[1,0].plot(ks, ch_vals, marker='o')
axs[1,0].set_title("CH (maior é melhor)")
axs[1,0].grid(True)

# SSE
axs[1,1].plot(ks, sse_vals, marker='o')
axs[1,1].set_title("SSE – Elbow")
axs[1,1].grid(True)

plt.tight_layout()
plt.show()


FASE 3 — ANÁLISE DE IMPORTÂNCIA DAS VARIÁVEIS
Objetivo corporativo: identificar variáveis que pioram o clustering, para remover e recalibrar o k ótimo.

In [ ]:
# ============================================================
# FASE 3 — Importância das variáveis
# ============================================================

loadings = pd.DataFrame(
    pca.components_.T,
    index=X_full.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

loadings["loading_strength"] = loadings.abs().mean(axis=1)

# ANOVA para clusterability
from sklearn.feature_selection import f_classif

f_vals, p_vals = f_classif(X_scaled_full, y_full)
anova = pd.DataFrame({
    "anova_fvalue": f_vals,
    "anova_pvalue": p_vals
}, index=X_full.columns)

df_importance = pd.concat([loadings["loading_strength"], anova], axis=1)
df_importance["anova_importance"] = 1 - (df_importance["anova_pvalue"] > 0.05).astype(int)
df_importance["final_score"] = (
      0.6 * df_importance["loading_strength"]
    + 0.4 * df_importance["anova_importance"]
)

df_importance.sort_values("final_score", ascending=False)


In [ ]:
# ============================================================
# PIPELINE COMPLETO — KMEANS (ADULT DATASET)
# Fluxo profissional: Dados > PCA > Seleção do k > Importância
# das variáveis > Remoção progressiva > Reavaliação > Modelos finais
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)
from sklearn.feature_selection import f_classif

# ------------------------------------------------------------
# 1. BASE, LIMPEZA E ENGENHARIA (usa df_adapt que você já criou)
# ------------------------------------------------------------

df_cluster = df_adapt.copy()

# Criar coluna binária apenas para interpretação
df_cluster["income_bin"] = (df_cluster["income"] == ">50K").astype(int)
df_cluster = df_cluster.drop(columns=["income"], errors="ignore")

# One-hot
df_encoded = pd.get_dummies(df_cluster, drop_first=True)

# Dataset para cluster (sem renda)
X = df_encoded.drop(columns=["income_bin"])
y = df_encoded["income_bin"]

print("Shape inicial:", X.shape)

# ------------------------------------------------------------
# 2. Escalonamento
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ------------------------------------------------------------
# 3. PCA retendo 90% da variância
# ------------------------------------------------------------
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)

print("PCA shape:", X_pca.shape)

# ------------------------------------------------------------
# 4. Seleção inicial do melhor k (k=2 a 20)
# ------------------------------------------------------------
ks = range(2, 21)

sil_vals = []
dbi_vals = []
ch_vals = []
sse_vals = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = km.fit_predict(X_pca)

    sil_vals.append(silhouette_score(X_pca, labels))
    dbi_vals.append(davies_bouldin_score(X_pca, labels))
    ch_vals.append(calinski_harabasz_score(X_pca, labels))
    sse_vals.append(km.inertia_)

df_k_initial = pd.DataFrame({
    "k": ks,
    "Silhouette": sil_vals,
    "Davies-Bouldin": dbi_vals,
    "Calinski-Harabasz": ch_vals,
    "SSE": sse_vals
})

print("\n=== MÉTRICAS INICIAIS (SEM REMOVER VARIÁVEIS) ===")
display(df_k_initial)

k_natural = df_k_initial.loc[df_k_initial["Silhouette"].idxmax(), "k"]
print("\nMelhor k natural (sem limpeza):", k_natural)

# ------------------------------------------------------------
# 5. Ranking de Importância das Variáveis
# ------------------------------------------------------------
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)
loadings["loading_strength"] = loadings.abs().mean(axis=1)

# ANOVA — clusterability
f_vals, p_vals = f_classif(X_scaled, y)
anova_df = pd.DataFrame({"p_value": p_vals}, index=X.columns)

# Score final
df_var_importance = pd.concat([loadings["loading_strength"], anova_df], axis=1)
df_var_importance["anova_importance"] = (df_var_importance["p_value"] < 0.05).astype(int)
df_var_importance["final_score"] = (
    0.6 * df_var_importance["loading_strength"] +
    0.4 * df_var_importance["anova_importance"]
)

df_var_importance = df_var_importance.sort_values("final_score", ascending=False)

print("\n=== RANKING DE IMPORTÂNCIA DAS VARIÁVEIS ===")
display(df_var_importance)

# ------------------------------------------------------------
# 6. Remover automaticamente variáveis menos importantes
#    (mantém apenas top 20 variáveis)
# ------------------------------------------------------------
top_vars = df_var_importance.head(20).index.tolist()
X_reduced = X[top_vars]

print("\nVariáveis selecionadas (top 20):")
print(top_vars)

# ------------------------------------------------------------
# 7. Reexecutar pipeline com variáveis reduzidas
# ------------------------------------------------------------
X_red_scaled = scaler.fit_transform(X_reduced)

pca_red = PCA(n_components=0.90)
X_red_pca = pca_red.fit_transform(X_red_scaled)

ks_red = range(2, 21)
sil_r, dbi_r, ch_r, sse_r = [], [], [], []

for k in ks_red:
    km = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = km.fit_predict(X_red_pca)

    sil_r.append(silhouette_score(X_red_pca, labels))
    dbi_r.append(davies_bouldin_score(X_red_pca, labels))
    ch_r.append(calinski_harabasz_score(X_red_pca, labels))
    sse_r.append(km.inertia_)

df_k_reduced = pd.DataFrame({
    "k": ks_red,
    "Silhouette": sil_r,
    "DBI": dbi_r,
    "CH": ch_r,
    "SSE": sse_r
})

print("\n=== MÉTRICAS APÓS REDUZIR VARIÁVEIS ===")
display(df_k_reduced)

k_reduced = df_k_reduced.loc[df_k_reduced["Silhouette"].idxmax(), "k"]
print("\nNovo k ideal após limpeza:", k_reduced)

# ------------------------------------------------------------
# 8. Modelos finais — k natural / k reduzido / k=4 / k=2
# ------------------------------------------------------------
modelos = {
    "k_natural": int(k_natural),
    "k_reduced": int(k_reduced),
    "k2": 2,
    "k4": 4
}

df_resultados = []

for nome, k_val in modelos.items():
    km = KMeans(n_clusters=k_val, random_state=42, n_init=100)
    labels_final = km.fit_predict(X_red_pca)

    sil = silhouette_score(X_red_pca, labels_final)
    dbi = davies_bouldin_score(X_red_pca, labels_final)
    ch = calinski_harabasz_score(X_red_pca, labels_final)

    df_resultados.append([nome, k_val, sil, dbi, ch])

df_resultados = pd.DataFrame(df_resultados,
    columns=["Modelo", "k", "Silhouette", "DBI", "CH"]
)

print("\n=== MODELOS FINAIS COMPARADOS ===")
display(df_resultados)

# ------------------------------------------------------------
# 9. Gráficos (Silhouette, CH, DBI, Elbow)
# ------------------------------------------------------------
fig, axs = plt.subplots(2, 2, figsize=(14,10))

axs[0,0].plot(ks_red, sil_r, marker="o")
axs[0,0].set_title("Silhouette (variáveis reduzidas)")

axs[0,1].plot(ks_red, dbi_r, marker="o")
axs[0,1].set_title("Davies-Bouldin")

axs[1,0].plot(ks_red, ch_r, marker="o")
axs[1,0].set_title("Calinski-Harabasz")

axs[1,1].plot(ks_red, sse_r, marker="o")
axs[1,1].set_title("SSE – Método do Cotovelo")

plt.tight_layout()
plt.show()


In [ ]:
labels_k4 = kmeans_k4.labels_
df_k4 = df_encoded.copy()
df_k4["cluster"] = labels_k4

print("labels_k4 recarregado com sucesso.")


In [ ]:
# ============================================================
# ANÁLISES PCA COMPLETAS – INTERPRETAÇÃO E GRÁFICOS AVANÇADOS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from matplotlib.patches import Ellipse
from sklearn.preprocessing import StandardScaler

# =======================================
# 1. PCA já calculado anteriormente
# =======================================
# X_pca: matriz transformada
# pca: objeto PCA ajustado
# labels_k4: rótulos do K-means k=4

pc_df = pd.DataFrame(X_pca[:, :2], columns=["PC1", "PC2"])
pc_df["cluster"] = labels_k4

# =======================================
# 2. Individuals Plot com rótulos
# =======================================
plt.figure(figsize=(10,7))
sns.scatterplot(
    data=pc_df, x="PC1", y="PC2", hue="cluster",
    palette="tab10", s=20, alpha=0.6
)
plt.title("Individuals Plot - PCA - Distribuição por Cluster (k=4)")
plt.axhline(0, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.show()

# =======================================
# 3. Elipses de confiança para cada cluster
# =======================================
from matplotlib.patches import Ellipse

def add_confidence_ellipse(x, y, ax, n_std=2, **kwargs):
    """
    Adiciona uma elipse de confiança baseada na matriz de covariância.
    """
    if len(x) < 2:
        return

    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)

    # ordenar autovalores
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))

    width, height = 2 * n_std * np.sqrt(vals)

    ellipse = Ellipse(
        (np.mean(x), np.mean(y)),
        width,
        height,
        angle=theta,        # <<< CORREÇÃO AQUI
        **kwargs
    )

    ax.add_patch(ellipse)


plt.figure(figsize=(10,7))
ax = plt.gca()

for cl in sorted(pc_df["cluster"].unique()):
    subset = pc_df[pc_df["cluster"] == cl]
    ax.scatter(subset["PC1"], subset["PC2"], s=20, alpha=0.5, label=f"Cluster {cl}")
    add_confidence_ellipse(subset["PC1"], subset["PC2"], ax,
                           edgecolor="black", facecolor="none")

plt.title("PCA – Elipses de Confiança por Cluster (k=4)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

# =======================================
# 4. Círculo de correlação das variáveis
# =======================================
loadings = pca.components_.T[:, :2]
features = X.columns.tolist()

plt.figure(figsize=(8,8))
plt.axhline(0, color='grey', linestyle='--')
plt.axvline(0, color='grey', linestyle='--')

for i, feature in enumerate(features):
    x, y = loadings[i]
    plt.arrow(0, 0, x, y, color='red', alpha=0.5)
    plt.text(x*1.1, y*1.1, feature, fontsize=9)

circle = plt.Circle((0,0), 1, facecolor='none', edgecolor='black')
plt.gca().add_artist(circle)

plt.title("PCA – Círculo de Correlação")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(False)
plt.show()

# =======================================
# 5. Gráfico de contribuição das variáveis
# =======================================
contrib_pc1 = (loadings[:,0]**2) / np.sum(loadings[:,0]**2)
contrib_pc2 = (loadings[:,1]**2) / np.sum(loadings[:,1]**2)

df_contrib = pd.DataFrame({
    "feature": features,
    "PC1": contrib_pc1,
    "PC2": contrib_pc2
}).sort_values("PC1", ascending=False)

plt.figure(figsize=(12,6))
sns.barplot(data=df_contrib.head(15), x="PC1", y="feature")
plt.title("Contribuição das Variáveis – PC1 (Top 15)")
plt.show()

plt.figure(figsize=(12,6))
sns.barplot(data=df_contrib.sort_values("PC2", ascending=False).head(15),
            x="PC2", y="feature")
plt.title("Contribuição das Variáveis – PC2 (Top 15)")
plt.show()

# =======================================
# 6. Heatmap dos loadings dos PCs
# =======================================
load_df = pd.DataFrame(loadings, index=features, columns=["PC1","PC2"])

plt.figure(figsize=(10,10))
sns.heatmap(load_df, annot=True, cmap="coolwarm", center=0)
plt.title("Heatmap dos Loadings – PC1 e PC2")
plt.show()


In [ ]:
# ============================================================
# PIPELINE COMPLETO — K-MEANS COM k = 2
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    silhouette_samples
)

from matplotlib.patches import Ellipse
import matplotlib.cm as cm
from math import pi

# ============================================================
# 1. Pré-processamento
# ============================================================

df_km = df_adapt.copy()

df_km["income_bin"] = (df_km["income"] == ">50K").astype(int)
df_km = df_km.drop(columns=["income"], errors="ignore")

df_encoded = pd.get_dummies(df_km, drop_first=True)

X = df_encoded.drop(columns=["income_bin"])
y = df_encoded["income_bin"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)

print(f"Formato original: {X_scaled.shape}")
print(f"Após PCA: {X_pca.shape}")

# ============================================================
# 2. K-means com k = 2
# ============================================================

k = 2
kmeans2 = KMeans(n_clusters=k, random_state=42, n_init=100)
labels_k2 = kmeans2.fit_predict(X_pca)

df_k2 = df_encoded.copy()
df_k2["cluster"] = labels_k2

print("\n=== Distribuição dos clusters (k=2) ===")
print(df_k2["cluster"].value_counts())

print("\n=== Proporção de >50K por cluster ===")
print(df_k2.groupby("cluster")["income_bin"].mean())

# ============================================================
# 3. Métricas globais
# ============================================================

sil = silhouette_score(X_pca, labels_k2)
dbi = davies_bouldin_score(X_pca, labels_k2)
ch  = calinski_harabasz_score(X_pca, labels_k2)

print("\n=== MÉTRICAS (k=2) ===")
print(f"Silhouette Score       : {sil:.4f}")
print(f"Davies-Bouldin Index   : {dbi:.4f}")
print(f"Calinski–Harabasz Index: {ch:.2f}")

# ============================================================
# 4. PCA interpretável — PC1 × PC2 com elipses
# ============================================================

def add_confidence_ellipse(x, y, ax, n_std=2, **kwargs):
    if len(x) < 2:
        return

    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)

    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)

    ellipse = Ellipse(
        (np.mean(x), np.mean(y)),
        width,
        height,
        angle=theta,
        **kwargs
    )
    ax.add_patch(ellipse)


pc_df = pd.DataFrame(X_pca[:, :2], columns=["PC1", "PC2"])
pc_df["cluster"] = labels_k2

plt.figure(figsize=(10,7))
ax = plt.gca()

for cl in sorted(pc_df["cluster"].unique()):
    subset = pc_df[pc_df["cluster"] == cl]
    ax.scatter(subset["PC1"], subset["PC2"], s=15, alpha=0.5, label=f"Cluster {cl}")
    add_confidence_ellipse(subset["PC1"], subset["PC2"], ax,
                           edgecolor="black", facecolor="none")

plt.title("PCA — PC1 × PC2 com Elipses (k=2)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

# ============================================================
# 5. Silhouette Plot
# ============================================================

sil_vals = silhouette_samples(X_pca, labels_k2)
sil_global = sil_vals.mean()

fig, ax1 = plt.subplots(figsize=(10, 7))
y_lower = 10

for i in range(k):
    ith_vals = sil_vals[labels_k2 == i]
    ith_vals.sort()

    size_cluster = ith_vals.shape[0]
    y_upper = y_lower + size_cluster

    color = cm.nipy_spectral(float(i) / k)
    ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_vals,
                      facecolor=color, edgecolor=color, alpha=0.7)

    ax1.text(-0.05, y_lower + 0.5 * size_cluster, str(i))
    y_lower = y_upper + 10

ax1.axvline(x=sil_global, color="red", linestyle="--")
ax1.set_title("Silhouette Plot (k=2)")
ax1.set_xlabel("Valor da Silhueta")
ax1.set_ylabel("Cluster")
plt.show()

# ============================================================
# 6. Heatmap — Médias padronizadas
# ============================================================

df_means_k2 = df_k2.groupby("cluster").mean()

scaler2 = StandardScaler()
df_means_scaled_k2 = pd.DataFrame(
    scaler2.fit_transform(df_means_k2),
    index=df_means_k2.index,
    columns=df_means_k2.columns
)

plt.figure(figsize=(14,10))
sns.heatmap(df_means_scaled_k2.T, cmap="coolwarm", center=0)
plt.title("Heatmap — Variáveis Padronizadas (k=2)")
plt.xlabel("Cluster")
plt.ylabel("Variáveis")
plt.tight_layout()
plt.show()

# ============================================================
# 7. Painel executivo — resumo dos perfis
# ============================================================

print("\n=== RESUMO EXECUTIVO DOS CLUSTERS (k=2) ===")
summary = pd.DataFrame({
    "n_obs": df_k2["cluster"].value_counts(),
    "pct_renda>50K": df_k2.groupby("cluster")["income_bin"].mean(),
    "idade_media": df_k2.groupby("cluster")["age"].mean(),
    "horas_semanais": df_k2.groupby("cluster")["hours.per.week"].mean(),
    "educacao_num": df_k2.groupby("cluster")["education.num"].mean(),
    "capital_gain_medio": df_k2.groupby("cluster")["capital.gain"].mean(),
    "capital_loss_medio": df_k2.groupby("cluster")["capital.loss"].mean(),
})

display(summary.round(2))


In [ ]:
# ============================================================
# REMOÇÃO AUTOMÁTICA DE VARIÁVEIS COM BAIXA CARGA NO PCA
# ============================================================

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Padronizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. PCA para 90% da variância
pca_full = PCA(n_components=0.90)
X_pca_full = pca_full.fit_transform(X_scaled)

# 3. Obter loadings (componentes)
loadings = pd.DataFrame(
    pca_full.components_[:2].T,
    index=X.columns,
    columns=["PC1","PC2"]
)

# 4. Selecionar variáveis com magnitude acima de threshold
threshold = 0.10
mask = (loadings.abs().max(axis=1) >= threshold)
selected_vars = loadings.index[mask].tolist()

print("Variáveis mantidas:", selected_vars)
print("Variáveis removidas:", list(loadings.index[~mask]))

# 5. Re-roda o pipeline apenas com variáveis relevantes
X_reduced = X[selected_vars]

# 6. Escalonar de novo
Xr_scaled = scaler.fit_transform(X_reduced)

# 7. PCA novamente
pca_r = PCA(n_components=0.90)
Xr_pca = pca_r.fit_transform(Xr_scaled)

print("Formato após filtrar variáveis:", Xr_pca.shape)

# ============================================================
# RODAR K-MEANS COM k=4 NOVAMENTE
# ============================================================

kmeans_r = KMeans(n_clusters=4, random_state=42, n_init=100)
labels_r = kmeans_r.fit_predict(Xr_pca)

# Métricas
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print("\n==== NOVAS MÉTRICAS APÓS REMOVER RUÍDO ====")
print("Silhouette:", silhouette_score(Xr_pca, labels_r))
print("DBI:", davies_bouldin_score(Xr_pca, labels_r))
print("CH:", calinski_harabasz_score(Xr_pca, labels_r))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import f_oneway, kruskal

# ============================================
# 1. Preparação das variáveis finais atuais
# ============================================
X_vars = X_reduced.columns.tolist()
X_sub = X_reduced.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sub)

# ============================================
# 2. Rodar PCA (90% da variância)
# ============================================
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)

loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    index=X_vars
)

# ============================================
# 3. K-means com k=4 (para consistência)
# ============================================
kmeans = KMeans(n_clusters=4, random_state=42, n_init=100)
labels = kmeans.fit_predict(X_pca)

print("Silhouette:", silhouette_score(X_pca, labels))

# ============================================
# 4. Importância via ANOVA / Kruskal
# ============================================

anova_results = {}
for var in X_vars:
    groups = [X_sub[var][labels == cl] for cl in np.unique(labels)]
    # kruskal funciona melhor com muitos outliers
    stat, pval = kruskal(*groups)
    anova_results[var] = pval

anova_df = pd.DataFrame.from_dict(anova_results, orient="index", columns=["p_value"])
anova_df = anova_df.sort_values("p_value")

print("\n=== P-valores (quanto menor, mais importante para separar clusters) ===")
display(anova_df)

# ============================================
# 5. Combinação PCA + ANOVA para ranking final
# ============================================
loading_strength = loadings.abs().mean(axis=1)   # média em todos os PCs
score_df = pd.DataFrame({
    "loading_strength": loading_strength,
    "anova_pvalue": anova_df["p_value"]
})

# normalizar p-values para colocar como peso
score_df["anova_importance"] = 1 - score_df["anova_pvalue"]

# score final ponderado: impacto PCA e importância ANOVA
score_df["final_score"] = (
    0.6 * score_df["loading_strength"] +
    0.4 * score_df["anova_importance"]
)

score_df = score_df.sort_values("final_score", ascending=False)
print("\n=== RANKING FINAL DE IMPORTÂNCIA DAS VARIÁVEIS ===")
display(score_df)


In [ ]:
vars_to_remove = ["race_3grp_Outras minorias", "country_continent_Ásia"]

X_final = X_reduced.drop(columns=vars_to_remove)

# re-escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# PCA 90%
pca = PCA(n_components=0.90)
X_pca_final = pca.fit_transform(X_scaled)

# novo kmeans
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=100)
labels_final = kmeans_final.fit_predict(X_pca_final)

# métricas
print("Silhouette:", silhouette_score(X_pca_final, labels_final))
print("DBI:", davies_bouldin_score(X_pca_final, labels_final))
print("CH:", calinski_harabasz_score(X_pca_final, labels_final))


In [ ]:
# ============================================================
# PIPELINE COMPLETO — KMEANS (ADULT DATASET)
# Fluxo profissional: Dados > PCA > Seleção do k > Importância
# das variáveis > Remoção progressiva > Reavaliação > Modelos finais
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)
from sklearn.feature_selection import f_classif

# ------------------------------------------------------------
# 1. BASE, LIMPEZA E ENGENHARIA (usa df_adapt que você já criou)
# ------------------------------------------------------------

df_cluster = df_adapt.copy()

# Criar coluna binária apenas para interpretação
df_cluster["income_bin"] = (df_cluster["income"] == ">50K").astype(int)
df_cluster = df_cluster.drop(columns=["income"], errors="ignore")

# One-hot
df_encoded = pd.get_dummies(df_cluster, drop_first=True)

# Dataset para cluster (sem renda)
X = df_encoded.drop(columns=["income_bin"])
y = df_encoded["income_bin"]

print("Shape inicial:", X.shape)

# ------------------------------------------------------------
# 2. Escalonamento
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ------------------------------------------------------------
# 3. PCA retendo 90% da variância
# ------------------------------------------------------------
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)

print("PCA shape:", X_pca.shape)

# ------------------------------------------------------------
# 4. Seleção inicial do melhor k (k=2 a 20)
# ------------------------------------------------------------
ks = range(2, 21)

sil_vals = []
dbi_vals = []
ch_vals = []
sse_vals = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = km.fit_predict(X_pca)

    sil_vals.append(silhouette_score(X_pca, labels))
    dbi_vals.append(davies_bouldin_score(X_pca, labels))
    ch_vals.append(calinski_harabasz_score(X_pca, labels))
    sse_vals.append(km.inertia_)

df_k_initial = pd.DataFrame({
    "k": ks,
    "Silhouette": sil_vals,
    "Davies-Bouldin": dbi_vals,
    "Calinski-Harabasz": ch_vals,
    "SSE": sse_vals
})

print("\n=== MÉTRICAS INICIAIS (SEM REMOVER VARIÁVEIS) ===")
display(df_k_initial)

k_natural = df_k_initial.loc[df_k_initial["Silhouette"].idxmax(), "k"]
print("\nMelhor k natural (sem limpeza):", k_natural)

# ------------------------------------------------------------
# 5. Ranking de Importância das Variáveis
# ------------------------------------------------------------
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)
loadings["loading_strength"] = loadings.abs().mean(axis=1)

# ANOVA — clusterability
f_vals, p_vals = f_classif(X_scaled, y)
anova_df = pd.DataFrame({"p_value": p_vals}, index=X.columns)

# Score final
df_var_importance = pd.concat([loadings["loading_strength"], anova_df], axis=1)
df_var_importance["anova_importance"] = (df_var_importance["p_value"] < 0.05).astype(int)
df_var_importance["final_score"] = (
    0.6 * df_var_importance["loading_strength"] +
    0.4 * df_var_importance["anova_importance"]
)

df_var_importance = df_var_importance.sort_values("final_score", ascending=False)

print("\n=== RANKING DE IMPORTÂNCIA DAS VARIÁVEIS ===")
display(df_var_importance)

# ------------------------------------------------------------
# 6. Remover automaticamente variáveis menos importantes
#    (mantém apenas top 20 variáveis)
# ------------------------------------------------------------
top_vars = df_var_importance.head(20).index.tolist()
X_reduced = X[top_vars]

print("\nVariáveis selecionadas (top 19):")
print(top_vars)

# ------------------------------------------------------------
# 7. Reexecutar pipeline com variáveis reduzidas
# ------------------------------------------------------------
X_red_scaled = scaler.fit_transform(X_reduced)

pca_red = PCA(n_components=0.90)
X_red_pca = pca_red.fit_transform(X_red_scaled)

ks_red = range(2, 21)
sil_r, dbi_r, ch_r, sse_r = [], [], [], []

for k in ks_red:
    km = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = km.fit_predict(X_red_pca)

    sil_r.append(silhouette_score(X_red_pca, labels))
    dbi_r.append(davies_bouldin_score(X_red_pca, labels))
    ch_r.append(calinski_harabasz_score(X_red_pca, labels))
    sse_r.append(km.inertia_)

df_k_reduced = pd.DataFrame({
    "k": ks_red,
    "Silhouette": sil_r,
    "DBI": dbi_r,
    "CH": ch_r,
    "SSE": sse_r
})

print("\n=== MÉTRICAS APÓS REDUZIR VARIÁVEIS ===")
display(df_k_reduced)

k_reduced = df_k_reduced.loc[df_k_reduced["Silhouette"].idxmax(), "k"]
print("\nNovo k ideal após limpeza:", k_reduced)

# ------------------------------------------------------------
# 8. Modelos finais — k natural / k reduzido / k=4 / k=2
# ------------------------------------------------------------
modelos = {
    "k_natural": int(k_natural),
    "k_reduced": int(k_reduced),
    "k2": 2,
    "k4": 4
}

df_resultados = []

for nome, k_val in modelos.items():
    km = KMeans(n_clusters=k_val, random_state=42, n_init=100)
    labels_final = km.fit_predict(X_red_pca)

    sil = silhouette_score(X_red_pca, labels_final)
    dbi = davies_bouldin_score(X_red_pca, labels_final)
    ch = calinski_harabasz_score(X_red_pca, labels_final)

    df_resultados.append([nome, k_val, sil, dbi, ch])

df_resultados = pd.DataFrame(df_resultados,
    columns=["Modelo", "k", "Silhouette", "DBI", "CH"]
)

print("\n=== MODELOS FINAIS COMPARADOS ===")
display(df_resultados)

# ------------------------------------------------------------
# 9. Gráficos (Silhouette, CH, DBI, Elbow)
# ------------------------------------------------------------
fig, axs = plt.subplots(2, 2, figsize=(14,10))

axs[0,0].plot(ks_red, sil_r, marker="o")
axs[0,0].set_title("Silhouette (variáveis reduzidas)")

axs[0,1].plot(ks_red, dbi_r, marker="o")
axs[0,1].set_title("Davies-Bouldin")

axs[1,0].plot(ks_red, ch_r, marker="o")
axs[1,0].set_title("Calinski-Harabasz")

axs[1,1].plot(ks_red, sse_r, marker="o")
axs[1,1].set_title("SSE – Método do Cotovelo")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# COMPARATIVO KMEANS — k = 2 e k = 4
# Base reduzida (top_vars)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples
import matplotlib.cm as cm

# -----------------------------
# 1. Preparação (X reduzido)
# -----------------------------
X_red = df_encoded[top_vars]
scaler = StandardScaler()
X_red_scaled = scaler.fit_transform(X_red)

pca_red = PCA(n_components=0.90)
X_pca_red = pca_red.fit_transform(X_red_scaled)

# -----------------------------
# 2. Função auxiliar de gráficos
# -----------------------------
def plot_cluster_results(k, X_pca):
    km = KMeans(n_clusters=k, random_state=42, n_init=100)
    labels = km.fit_predict(X_pca)

    # métricas
    sil = silhouette_score(X_pca, labels)
    dbi = davies_bouldin_score(X_pca, labels)
    ch = calinski_harabasz_score(X_pca, labels)

    print(f"\n==============================")
    print(f"RESULTADOS — K = {k}")
    print("==============================")
    print(f"Silhouette: {sil:.4f}")
    print(f"Davies-Bouldin: {dbi:.4f}")
    print(f"Calinski-Harabasz: {ch:.4f}")
    print("\nDistribuição dos clusters:")
    print(pd.Series(labels).value_counts())

    # PCA plot (PC1 x PC2)
    df_plot = pd.DataFrame({
        "PC1": X_pca[:, 0],
        "PC2": X_pca[:, 1],
        "cluster": labels
    })

    plt.figure(figsize=(7,5))
    sns.scatterplot(data=df_plot, x="PC1", y="PC2", hue="cluster",
                    palette="tab10", alpha=0.7)
    plt.title(f"PCA (PC1 × PC2) — K = {k}")
    plt.show()

    # Silhouette plot
    sil_vals = silhouette_samples(X_pca, labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    y_lower = 10

    for i in range(k):
        values = sil_vals[labels == i]
        values.sort()
        size = values.shape[0]
        y_upper = y_lower + size

        color = cm.nipy_spectral(float(i) / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, values,
                         facecolor=color, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size, str(i))
        y_lower = y_upper + 10

    ax.axvline(sil, color="red", linestyle="--")
    ax.set_title(f"Silhouette Plot — K = {k}")
    plt.show()

    # Heatmap de médias padronizadas
    df_clustered = X_red.copy()
    df_clustered["cluster"] = labels

    df_means = df_clustered.groupby("cluster").mean()
    df_means_scaled = pd.DataFrame(
        StandardScaler().fit_transform(df_means),
        index=df_means.index,
        columns=df_means.columns
    )

    plt.figure(figsize=(10, 8))
    sns.heatmap(df_means_scaled.T, cmap="coolwarm", center=0)
    plt.title(f"Heatmap — Médias Padronizadas (k = {k})")
    plt.show()

    return sil, dbi, ch


# -----------------------------
# 3. Executar análises para k=2 e k=4
# -----------------------------
results = {}
results["k2"] = plot_cluster_results(2, X_pca_red)
results["k4"] = plot_cluster_results(4, X_pca_red)

print("\n\nComparativo Final — Desempenho")
df_compare = pd.DataFrame(results, index=["Silhouette", "DBI", "CH"])
display(df_compare)


In [ ]:
# =============================================================
# GRÁFICOS DE INTERPRETAÇÃO PARA K-MEANS (qualquer K)
# =============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from matplotlib import cm
from math import pi

# -------------------------------------------------------------
# 1. Definir valor de k
# -------------------------------------------------------------
k_val = 4   # coloque 2 ou 4 conforme queira analisar

# -------------------------------------------------------------
# 2. Rodar kmeans com PCA reduzido (já criado)
# -------------------------------------------------------------
kmeans = KMeans(n_clusters=k_val, random_state=42, n_init=100)
labels = kmeans.fit_predict(X_red_pca)

df_plot = pd.DataFrame({
    "PC1": X_red_pca[:, 0],
    "PC2": X_red_pca[:, 1],
    "cluster": labels
})

# -------------------------------------------------------------
# 3. PCA – Scatterplot PC1 × PC2 (interpretação da separação)
# -------------------------------------------------------------
plt.figure(figsize=(9,6))
sns.scatterplot(
    data=df_plot, x="PC1", y="PC2", hue="cluster",
    palette="tab10", alpha=0.6
)
plt.title(f"PCA (PC1 × PC2) — K = {k_val}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.show()

# -------------------------------------------------------------
# 4. Silhouette Plot
# -------------------------------------------------------------
sil_vals = silhouette_samples(X_red_pca, labels)
sil_avg = silhouette_score(X_red_pca, labels)

fig, ax = plt.subplots(figsize=(9,6))

y_lower = 10
for i in range(k_val):
    ith = sil_vals[labels == i]
    ith.sort()
    size_i = ith.shape[0]
    y_upper = y_lower + size_i

    color = cm.nipy_spectral(float(i) / k_val)
    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, ith, facecolor=color, edgecolor=color, alpha=0.7
    )
    ax.text(-0.05, y_lower + size_i/2, str(i))
    y_lower = y_upper + 10

ax.axvline(x=sil_avg, color="red", linestyle="--")
ax.set_title(f"Silhouette Plot — K = {k_val}")
ax.set_xlabel("Valor de Silhouette")
ax.set_ylabel("Cluster")
plt.show()

# -------------------------------------------------------------
# 5. Heatmap – médias padronizadas por cluster
# -------------------------------------------------------------
df_c = df_encoded[top_vars].copy()
df_c["cluster"] = labels

df_means = df_c.groupby("cluster").mean()

# padronizar médias para heatmap
df_scaled = (df_means - df_means.mean()) / df_means.std()

plt.figure(figsize=(12,9))
sns.heatmap(df_scaled.T, cmap="coolwarm", center=0)
plt.title(f"Heatmap — Médias Padronizadas (K = {k_val})")
plt.xlabel("Cluster")
plt.ylabel("Variáveis")
plt.tight_layout()
plt.show()

# -------------------------------------------------------------
# 6. Radar Chart dos perfis médios
# (ótimo para comparar clusters em “forma”)
# -------------------------------------------------------------
def radar_chart_means(df_means, k_val):
    categories = list(df_means.columns)
    N = len(categories)

    for cluster_id in range(k_val):
        values = df_means.loc[cluster_id].values
        values = np.append(values, values[0])

        angles = [n / float(N) * 2 * pi for n in range(N)]
        angles += angles[:1]

        plt.figure(figsize=(6,6))
        ax = plt.subplot(111, polar=True)
        ax.set_theta_offset(pi/2)
        ax.set_theta_direction(-1)

        plt.xticks(angles[:-1], categories, fontsize=8)
        ax.plot(angles, values, linewidth=2)
        ax.fill(angles, values, alpha=0.2)
        plt.title(f"Radar Chart — Cluster {cluster_id}")
        plt.tight_layout()
        plt.show()

radar_chart_means(df_means, k_val)

# -------------------------------------------------------------
# 7. Barplot das variáveis que mais diferenciam clusters
# -------------------------------------------------------------
cluster_std = df_means.std(axis=0).sort_values(ascending=False)
top_diff = cluster_std.head(10)

plt.figure(figsize=(8,6))
sns.barplot(x=top_diff.values, y=top_diff.index, palette="viridis")
plt.title(f"Variáveis que mais diferenciam clusters (K = {k_val})")
plt.xlabel("Desvio padrão entre clusters (maior = mais discriminante)")
plt.tight_layout()
plt.show()


In [ ]:
# ======================================================
# TOP VARIÁVEIS POR CLUSTER
# ======================================================

n = 5  # número de variáveis por cluster

for cl in df_means.index:
    dif = (df_means.loc[cl] - df_means.mean())  # diferença em relação à média global
    dif = dif.abs().sort_values(ascending=False).head(n)

    plt.figure(figsize=(8,4))
    sns.barplot(x=dif.values, y=dif.index, palette="crest")
    plt.title(f"Variáveis que mais destacam o cluster {cl} (K = {k_val})")
    plt.xlabel("Distância da média geral")
    plt.tight_layout()
    plt.show()


In [ ]:
for k_val in [2, 4]:
    print(f"\n==================== K = {k_val} ====================")

    # 1. Recalcula clusters
    km = KMeans(n_clusters=k_val, random_state=42, n_init=100)
    labels = km.fit_predict(X_red_pca)

    df_c = df_encoded[top_vars].copy()
    df_c["cluster"] = labels
    df_means = df_c.groupby("cluster").mean()

    # 2. Barplot - Todas as variáveis
    cluster_std_all = df_means.std(axis=0).sort_values(ascending=False)
    plt.figure(figsize=(10,14))
    sns.barplot(x=cluster_std_all.values, y=cluster_std_all.index, palette="viridis")
    plt.title(f"Variáveis que mais diferenciam clusters (K = {k_val})")
    plt.xlabel("Desvio padrão entre clusters")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# CORREÇÃO — recuperar valores corretos para cálculo
# ============================================================


df_k4 = df_encoded.copy()
df_k4["cluster"] = labels_k4

print("df_k4 reconstruído. Clusters disponíveis:")
print(df_k4["cluster"].value_counts())

# Escolher k desejado (altere para 2, 4, 20...)
k_value = 4   # coloque 2 se quiser testar k = 2

km = KMeans(n_clusters=k_value, random_state=42, n_init=100)
labels = km.fit_predict(X_pca)
centroids = km.cluster_centers_

# ============================================================
# DISTÂNCIA INTRA-CLUSTER — média de distância ao centróide
# ============================================================

from scipy.spatial.distance import cdist
dist_intra = []

for cl in range(k_value):
    points = X_pca[labels == cl]
    centroid = centroids[cl]
    dist_intra.append(np.mean(cdist(points, [centroid])))

dist_intra = pd.Series(dist_intra, name="Intra-cluster distance")
print("\nDistância média intra-cluster (quanto menor, mais compacto):")
display(dist_intra)

# ============================================================
# DISTÂNCIA ENTRE CENTRÓIDES — separação dos clusters
# ============================================================

dist_matrix = cdist(centroids, centroids, metric="euclidean")
dist_matrix_df = pd.DataFrame(dist_matrix,
                              index=[f"Cluster {i}" for i in range(k_value)],
                              columns=[f"Cluster {i}" for i in range(k_value)])

print("\nMatriz de distâncias entre centróides (quanto maior, mais separados):")
display(dist_matrix_df)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.spatial import ConvexHull

# ============================================================
# 1. Rodar K-means com o k desejado
# ============================================================
k_value = 4   # coloque 2 para o k = 2
km = KMeans(n_clusters=k_value, random_state=42, n_init=100)
labels = km.fit_predict(X_pca)

pc_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": labels
})

# ============================================================
# 2. Função para desenhar o “convex hull”
# ============================================================
def plot_convex_hull(points, color, ax):
    if len(points) < 3:
        return
    hull = ConvexHull(points)
    hull_pts = points[hull.vertices]
    ax.fill(hull_pts[:,0], hull_pts[:,1], alpha=0.2, color=color)

# Paleta
colors = ["#E74C3C", "#27AE60", "#3498DB", "#9B59B6", "#F1C40F", "#1ABC9C"]

# ============================================================
# 3. Plot estilo fviz_cluster
# ============================================================
plt.figure(figsize=(10,7))
ax = plt.gca()

for cl in range(k_value):
    subset = pc_df[pc_df["cluster"] == cl]
    pts = subset[["PC1", "PC2"]].values

    # desenha o cluster
    ax.scatter(subset["PC1"], subset["PC2"],
               s=50, color=colors[cl], label=f"Cluster {cl}", alpha=0.7)

    # convex hull
    plot_convex_hull(pts, colors[cl], ax)

# rótulos opcionais
# for i, row in pc_df.iterrows():
#     ax.text(row["PC1"], row["PC2"], str(i), fontsize=4)

plt.axhline(0, color="black", linewidth=0.6, linestyle="--")
plt.axvline(0, color="black", linewidth=0.6, linestyle="--")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title(f"Cluster Plot estilo fviz_cluster (k = {k_value})")
plt.legend()
plt.show()


In [ ]:
# ============================================================
# PROPORÇÃO DE >50K POR CLUSTER (k = 4)
# ============================================================

# labels = resultados do KMeans
# df_encoded = base completa com income_bin

df_temp = df_encoded.copy()
df_temp["cluster"] = labels

# proporção de renda alta
prop_income = df_temp.groupby("cluster")["income_bin"].mean()

print("=== Proporção de indivíduos >50K por cluster ===")
print(prop_income)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.spatial import ConvexHull

# ============================================================
# 1. Rodar K-means com o k = 2
# ============================================================
k_value = 2     # AGORA USANDO K = 2

km = KMeans(n_clusters=k_value, random_state=42, n_init=100)
labels = km.fit_predict(X_pca)

pc_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": labels
})

# ============================================================
# 2. Função para desenhar o convex hull
# ============================================================
def plot_convex_hull(points, color, ax):
    if len(points) < 3:
        return
    hull = ConvexHull(points)
    hull_pts = points[hull.vertices]
    ax.fill(hull_pts[:,0], hull_pts[:,1], alpha=0.20, color=color)

# Paleta de cores
colors = ["#E74C3C", "#3498DB"]

# ============================================================
# 3. Plot estilo fviz_cluster
# ============================================================
plt.figure(figsize=(10,7))
ax = plt.gca()

for cl in range(k_value):
    subset = pc_df[pc_df["cluster"] == cl]
    pts = subset[["PC1", "PC2"]].values

    ax.scatter(
        subset["PC1"], subset["PC2"],
        s=50, color=colors[cl], label=f"Cluster {cl}", alpha=0.7
    )

    plot_convex_hull(pts, colors[cl], ax)

plt.axhline(0, color="black", linewidth=0.6, linestyle="--")
plt.axvline(0, color="black", linewidth=0.6, linestyle="--")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("Cluster Plot estilo fviz_cluster (k = 2)")
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
# Reconstruir DF_K4 corretamente
k_fixed = 4
kmeans_k4 = KMeans(n_clusters=k_fixed, random_state=42, n_init=100)
labels_k4 = kmeans_k4.fit_predict(X_red_pca)

df_k4 = df_encoded.copy()
df_k4["cluster"] = labels_k4

# df_k4 = base com cluster (já usado no seu pipeline)
# X_reduced = base numérica (somente top 20)
# y = income_bin

df_cluster_profile = df_k4.groupby("cluster")[X_reduced.columns].mean()

# Padronizar para comparação relativa
scaler = StandardScaler()
df_cluster_profile_std = pd.DataFrame(
    scaler.fit_transform(df_cluster_profile),
    index=df_cluster_profile.index,
    columns=df_cluster_profile.columns
)

print("=== MÉDIAS PADRONIZADAS POR CLUSTER ===")
display(df_cluster_profile_std)

# Variáveis que mais diferenciam
importance = df_cluster_profile_std.abs().T
top_features_by_cluster = {}

for cl in df_cluster_profile_std.index:
    top_vars = importance[cl].sort_values(ascending=False).head(10)
    top_features_by_cluster[cl] = top_vars.index.tolist()

print("\n=== VARIÁVEIS QUE MAIS DIFERENCIAM CADA CLUSTER ===")
for cl, feats in top_features_by_cluster.items():
    print(f"Cluster {cl}: {feats}")

# Interpretação automática
def interpretar_cluster(cluster_id):
    print("\n===============================")
    print(f"INTERPRETAÇÃO DO CLUSTER {cluster_id}")
    print("===============================")

    perfil = df_cluster_profile_std.loc[cluster_id]
    high = perfil[perfil > 0.7].sort_values(ascending=False)
    low = perfil[perfil < -0.7].sort_values()

    print("\nVariáveis muito acima da média:")
    display(high)

    print("\nVariáveis muito abaixo da média:")
    display(low)

    # Income médio
    income_mean = df_k4[df_k4["cluster"] == cluster_id]["income_bin"].mean()
    print(f"\nProporção de renda >50K neste cluster: {income_mean:.3f}")

# Gerar interpretação para k=4
for cl in sorted(df_k4.cluster.unique()):
    interpretar_cluster(cl)


In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df_cluster_profile_std, cmap="coolwarm", center=0,
            linewidths=.5, annot=False)
plt.title("Heatmap — Médias Padronizadas por Cluster")
plt.xlabel("Variáveis")
plt.ylabel("Cluster")
plt.show()


In [ ]:
import numpy as np

def plot_radar_cluster(cluster_id):
    data = df_cluster_profile_std.loc[cluster_id]
    labels = data.index.tolist()
    values = data.values
    values = np.concatenate((values, [values[0]]))  # fechar o gráfico
    angles = np.linspace(0, 2*np.pi, len(values))

    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=8, rotation=45)
   # plt.title(f"Radar Chart — Cluster {cluster_id}")
    plt.show()

# Exemplo
plot_radar_cluster(0)
plot_radar_cluster(1)
plot_radar_cluster(2)
plot_radar_cluster(3)


In [ ]:
# ============================================================
# K = 2 (FINAL) — GARANTIR labels_k2 SEM RECALCULAR TUDO
# ============================================================

from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Garantir que labels_k2 exista
if "labels_k2" not in globals():
    km2 = KMeans(n_clusters=2, random_state=42, n_init=50)
    labels_k2 = km2.fit_predict(X_red_pca)
    print("K=2 recalculado apenas porque labels_k2 não existia.")
else:
    print("K=2 reaproveitado (labels_k2 já existia).")

# 2. Construir dataframe de análise
df_k2 = X_reduced.copy()
df_k2["cluster"] = labels_k2
df_k2["income_bin"] = y_full.values   # agora combina


# 3. Perfil médio dos clusters
df_cluster_profile_k2 = df_k2.groupby("cluster")[X_reduced.columns].mean()

# 4. Padronizar para comparação
from sklearn.preprocessing import StandardScaler
scaler_prof = StandardScaler()
df_cluster_profile_std_k2 = pd.DataFrame(
    scaler_prof.fit_transform(df_cluster_profile_k2),
    columns=df_cluster_profile_k2.columns,
    index=df_cluster_profile_k2.index
)

print("\n=== PERFIL PADRONIZADO — K=2 ===")
display(df_cluster_profile_std_k2)

# 5. Proporção de renda >50k por cluster
print("\n=== PROPORÇÃO DE >50K POR CLUSTER — K=2 ===")
print(df_k2.groupby("cluster")["income_bin"].mean())

# ============================================================
# GRÁFICO RADAR — AUTOMÁTICO PARA K=2
# ============================================================

import numpy as np

def plot_radar_cluster_k2(cluster_id):
    data = df_cluster_profile_std_k2.loc[cluster_id]
    labels = data.index.tolist()
    values = data.values
    values = np.concatenate((values, [values[0]]))
    angles = np.linspace(0, 2*np.pi, len(values))

    plt.figure(figsize=(8,8))
    ax = plt.subplot(111, polar=True)
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=7)
    #plt.title(f"Radar Chart — Cluster {cluster_id}")
    plt.show()

# gerar gráficos
plot_radar_cluster_k2(0)
plot_radar_cluster_k2(1)


## Interpretação técnica e limitações

O notebook emprega K-means na base Adult Income, incluindo transformações de variáveis, PCA, métricas de agrupamento e análise dos perfis dos clusters.

**Resultados registrados:** a preparação imprime matriz final com 32.561 linhas e 27 colunas. Em uma configuração com quatro clusters, os tamanhos registrados são 12.436, 5.263, 14.703 e 159 observações; a distribuição muito desigual, em especial o grupo com 159 casos, demanda interpretação cautelosa. Em uma configuração com dois clusters, as quantidades registradas são 19.516 e 13.045. Uma das avaliações após redução de variáveis reporta coeficiente de silhueta de aproximadamente 0,1752, indicando separação relativamente fraca naquele ajuste.

**Limitações:** K-means não confirma a existência de grupos naturais; o número de clusters depende da codificação, padronização, escolha de atributos e critérios de avaliação. Uma proporção de renda alta igual a 100% em um cluster pequeno não demonstra previsibilidade fora da amostra nem causalidade. Variáveis categóricas codificadas e PCA exigem justificativa para a distância euclidiana. O conteúdo interpreta saídas registradas, sem reexecução.